# ES 335 Project — Retail Sales Forecasting and Feature Analysis
**Team:** Krishn Kumar (25110172), Arush Banerjee (25110056)  
**Data source:** Kaggle Store Sales – Time Series Forecasting (problem and dataset only)  
**Track:** ES 335 Undergraduate

## Research question
How well do classical regression and tree-based models forecast daily sales across store–product-family series over a 16-day horizon, and which groups of available features improve out-of-time forecasting performance?

**Observation unit:** one store × product family × date. **Target:** non-negative daily sales. **Prediction origin:** the day before a 16-day forecast window. Sales observed inside the forecast window are unavailable.

This is an independent course project, not a Kaggle leaderboard submission. Models: Seasonal naive (Base model), OLS, Ridge, Lasso, Decision Tree, Random Forest, KNN, and LightGBM when available.

## Forecast-time information boundary
| Information | Available at forecast origin? | Treatment |
|---|---|---|
| Historical sales before the forecast window | Yes | Lags and rolling summaries shifted by at least 16 days |
| Calendar dates in the next 16 days | Yes | Calendar and cyclical features |
| Store metadata | Yes, assumed static | Merged from store metadata |
| `onpromotion` values in supplied test rows | Only if announced in advance | Used as known-future input under this assumption |
| Actual sales during the 16-day forecast window | No | Never used as a predictor |
| Future oil prices or future transactions | Not established as known | Excluded |

**Important assumption:** the competition test file provides promotion counts for forecast dates. We treat them as known at prediction time. If promotions were not announced in advance in a real deployment, the feature set would need to change.

## Results at a glance
The quantitative summary is generated near the end after the final holdout section runs. Re-run all cells from top to bottom to refresh the values; do not copy numerical results from another project.

The notebook follows the ES 335 rubric: problem, data, preprocessing, baseline/methods, evaluation, results/errors/limitations, and reproducibility.


## 0. Data access and computational environment

The experiments use the Store Sales – Time Series Forecasting files `train.csv`, `test.csv`, `stores.csv`, and `holidays_events.csv`. The notebook searches the standard Kaggle input locations and can also be configured to read the files from a local directory through `DATA_DIR`.

The raw competition data and any access credentials are kept outside the public project repository. The notebook records the Python and library environment used for reproducibility.

In [ ]:
from pathlib import Path
import glob, json, platform, warnings
from importlib.metadata import PackageNotFoundError, version as package_version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED, HORIZON, N_FOLDS = 42, 16, 3
MAX_TRAIN_ROWS = 80_000  # deterministic cap for sklearn model fitting; disclose in report

def locate_data():
    candidates = [
        Path("/kaggle/input/store-sales-time-series-forecasting"),
        Path("/kaggle/input/competitions/store-sales-time-series-forecasting"),
        Path("./store-sales-time-series-forecasting"), Path(".")
    ]
    for p in candidates:
        if all((p/f).exists() for f in ["train.csv","test.csv","stores.csv","holidays_events.csv"]):
            return p
    for f in glob.glob("/kaggle/input/**/train.csv", recursive=True):
        p=Path(f).parent
        if (p/"stores.csv").exists() and (p/"holidays_events.csv").exists(): return p
    raise FileNotFoundError("Attach the Kaggle dataset or set DATA_DIR to its CSV directory.")

DATA_DIR=locate_data()
train=pd.read_csv(DATA_DIR/"train.csv",parse_dates=["date"])
test=pd.read_csv(DATA_DIR/"test.csv",parse_dates=["date"])
stores=pd.read_csv(DATA_DIR/"stores.csv")
holidays=pd.read_csv(DATA_DIR/"holidays_events.csv",parse_dates=["date"])
print("Python",platform.python_version(),"Data",DATA_DIR)
print("train",train.shape,"test",test.shape,"stores",stores.shape,"holidays",holidays.shape)
display(train.head())
PACKAGE_VERSIONS = {}
for package in ["numpy", "pandas", "matplotlib", "scikit-learn", "lightgbm", "jupyter"]:
    try:
        PACKAGE_VERSIONS[package] = package_version(package)
    except PackageNotFoundError:
        PACKAGE_VERSIONS[package] = None
print("Installed package versions:", PACKAGE_VERSIONS)


## 1. Problem formulation and data understanding

The observation unit is one store × product family × date. The target is daily sales, and `id` is retained only as a row identifier rather than used as a predictor. The analysis examines temporal coverage, missing values, duplicate observations, zero and non-negative sales, the target distribution, and variation across product families.

The training data contain 3,000,888 observations from 1 January 2013 through 15 August 2017. The competition test period covers 16–31 August 2017. The dataset contains 54 stores and 33 product families. No missing values or duplicate store–family–date observations were found in the inspected training data; approximately 31.3% of sales observations are zero.

### Main-field data dictionary

| Field / source | Meaning | Treatment |
|---|---|---|
| date | Calendar date of the store-family observation | Calendar and horizon features |
| store_nbr | Store identifier | Series key and store-metadata join |
| family | Product-family category | Series key and category context |
| sales | Daily sales | Non-negative target, transformed with log1p |
| onpromotion | Promotion count for a store-family-date | Used under the stated known-in-advance assumption |
| stores.csv | City, state, store type and cluster | Store context |
| holidays_events.csv | Date, locale, locale name, type and transfer status | National, regional and local holiday flags |
| id | Competition row identifier | Traceability only; excluded from features |

The competition test panel is expected to contain 16 dates × 54 stores × 33 families = 28,512 rows. The test panel is checked for duplicate keys, date coverage and known store/family identifiers.


In [ ]:
required={"id","date","store_nbr","family","sales","onpromotion"}
assert required.issubset(train.columns), required-set(train.columns)
assert {"id","date","store_nbr","family","onpromotion"}.issubset(test.columns)
assert not train.duplicated(["date","store_nbr","family"]).any()
assert not test.duplicated(["date","store_nbr","family"]).any()
assert test["date"].nunique()==HORIZON, "Competition test should contain exactly 16 forecast dates."
expected_test_rows=train.store_nbr.nunique()*train.family.nunique()*HORIZON
assert len(test)==expected_test_rows, f"Expected {expected_test_rows:,} competition test rows, got {len(test):,}."
assert test.groupby("date").size().eq(train.store_nbr.nunique()*train.family.nunique()).all(), "At least one test date has incomplete panel coverage."
assert set(test.store_nbr).issubset(set(stores.store_nbr)), "Test data contains an unknown store identifier."
assert set(test.family).issubset(set(train.family)), "Test data contains an unknown product family."
print("Train dates:",train.date.min(),"to",train.date.max())
print("Test dates:",test.date.min(),"to",test.date.max())
print("Stores:",train.store_nbr.nunique(),"Families:",train.family.nunique())
print("Missing values:"); display(train.isna().sum().to_frame("count"))
print("Negative sales:",(train.sales<0).sum(),"Zero fraction:",(train.sales==0).mean())
fig,ax=plt.subplots(1,2,figsize=(12,4))
train.sales.clip(upper=train.sales.quantile(.99)).hist(bins=60,ax=ax[0]); ax[0].set_title("Sales (display capped at p99)")
np.log1p(train.sales.clip(lower=0)).hist(bins=60,ax=ax[1]); ax[1].set_title("log1p(sales)")
plt.tight_layout(); plt.show()
display(train.groupby("family").sales.agg(["count","mean","median","std"]).sort_values("mean",ascending=False).head(12))
print("Test duplicate keys:", test.duplicated(["date","store_nbr","family"]).sum())
print("Test dates:", test.date.nunique(), "| expected rows:", expected_test_rows)


In [ ]:
daily=train.groupby("date").sales.sum().sort_index()
daily.plot(figsize=(14,4),title="Total daily sales across stores and families"); plt.ylabel("Sales"); plt.show()
dow=train.assign(dow=train.date.dt.dayofweek).groupby("dow").sales.mean()
dow.plot(kind="bar",figsize=(7,3),title="Mean sales by weekday (0=Monday)"); plt.show()

## 2. Preprocessing and feature engineering

The forecasting setup uses a fixed 16-day horizon. At each prediction origin, sales observed inside the forecast window are unavailable. To respect this information boundary, sales lags begin at 16 days and rolling summaries are shifted by 16 days. These features are calculated independently for each store–product-family series.

The feature set combines calendar variables, cyclical calendar representations, store attributes, holiday indicators, promotion counts, and historical sales summaries. Actual future sales are not used. Future oil prices and transactions are excluded because their availability at the forecast origin is not established. The test-period promotion values are treated as known inputs, which is an explicit forecasting assumption.

In [ ]:
if (train.sales<0).any(): raise ValueError("Negative target found: investigate before log1p.")
def calendar_features(d):
    x=d.copy(); dt=pd.to_datetime(x.date)
    x["dow"]=dt.dt.dayofweek; x["day"]=dt.dt.day; x["month"]=dt.dt.month
    x["week"]=dt.dt.isocalendar().week.astype(int); x["quarter"]=dt.dt.quarter; x["year"]=dt.dt.year
    x["weekend"]=(x.dow>=5).astype(int)
    x["dow_sin"]=np.sin(2*np.pi*x.dow/7); x["dow_cos"]=np.cos(2*np.pi*x.dow/7)
    x["month_sin"]=np.sin(2*np.pi*(x.month-1)/12); x["month_cos"]=np.cos(2*np.pi*(x.month-1)/12)
    return x

def build_features(train,test):
    a=train.copy(); b=test.copy()
    a["log_sales"]=np.log1p(a.sales.clip(lower=0))
    b["sales"]=np.nan; b["log_sales"]=np.nan
    x=pd.concat([a,b],ignore_index=True,sort=False).sort_values(["store_nbr","family","date"]).reset_index(drop=True)
    x=x.merge(stores,on="store_nbr",how="left",validate="many_to_one")
    h=holidays[holidays.transferred.fillna(False)==False].copy()
    h["flag"]=1
    nat=h[h.locale=="National"].groupby("date").flag.max().rename("national_holiday")
    x=x.merge(nat,left_on="date",right_index=True,how="left")
    x["national_holiday"]=x.national_holiday.fillna(0).astype(int)
    reg=h[h.locale=="Regional"][["date","locale_name"]].drop_duplicates().rename(columns={"locale_name":"state"})
    reg["regional_holiday"]=1
    x=x.merge(reg,on=["date","state"],how="left")
    x["regional_holiday"]=x.regional_holiday.fillna(0).astype(int)
    loc=h[h.locale=="Local"][["date","locale_name"]].drop_duplicates().rename(columns={"locale_name":"city"})
    loc["local_holiday"]=1
    x=x.merge(loc,on=["date","city"],how="left")
    x["local_holiday"]=x.local_holiday.fillna(0).astype(int)
    x=calendar_features(x)
    grp=x.groupby(["store_nbr","family"],sort=False).log_sales
    for lag in [16,23,30,37,44,51,58,65,72,79,86,93,100,107,114,121,128,135,142,149,156,163,170,177,184,191,198,205,212,219,226,233,240,247,254,261,268,275,282,289,296,303,310,317,324,331,338,345,352,359,366,373]:
        x[f"lag_{lag}"]=grp.shift(lag)
    for w in [7,14,28]:
        x[f"roll_mean_{w}"]=x.groupby(["store_nbr","family"]).log_sales.transform(lambda s:s.shift(HORIZON).rolling(w,min_periods=3).mean())
        x[f"roll_std_{w}"]=x.groupby(["store_nbr","family"]).log_sales.transform(lambda s:s.shift(HORIZON).rolling(w,min_periods=3).std())
    x["promo_log1p"]=np.log1p(x.onpromotion.fillna(0).clip(lower=0))
    return x

data=build_features(train,test)
dev=data[data.sales.notna()].copy()
future=data[data.sales.isna()].copy()
print("Development rows",dev.shape,"future rows",future.shape)

## 3. Baseline and time-aware evaluation

Model performance is assessed with three rolling-origin development folds, each containing a 16-day validation window. For every fold, the model is fitted only on observations dated before that window. The final 16 labelled days are held out for a separate evaluation after model selection.

The Seasonal naive (Base model) uses the latest same-weekday observation available at the forecast origin. If that lookup is missing, it falls back to the store-family historical median. This provides a transparent weekly-pattern reference without using information from inside the forecast window.


In [ ]:
def rmsle(y,p):
    y=np.asarray(y,float); p=np.clip(np.asarray(p,float),0,None)
    return float(np.sqrt(np.mean((np.log1p(y)-np.log1p(p))**2)))
def mae(y,p): return float(np.mean(np.abs(np.asarray(y)-np.asarray(p))))

def naive_forecast(history,target,origin):
    lookup=history.set_index(["date","store_nbr","family"]).sales
    h=(target.date-pd.Timestamp(origin)).dt.days
    src=target.date-pd.to_timedelta(7*np.ceil(h/7).astype(int),unit="D")
    keys=pd.MultiIndex.from_arrays([src,target.store_nbr,target.family],names=["date","store_nbr","family"])
    p=lookup.reindex(keys).to_numpy(dtype=float)
    med=history.groupby(["store_nbr","family"]).sales.median()
    miss=np.isnan(p)
    if miss.any():
        mk=pd.MultiIndex.from_arrays([target.loc[miss,"store_nbr"],target.loc[miss,"family"]])
        p[miss]=med.reindex(mk).to_numpy()
    return np.nan_to_num(p,nan=0).clip(0)

dates=np.sort(dev.date.unique())
hold_dates=dates[-HORIZON:]
hold_start=pd.Timestamp(hold_dates.min())
folds=[]
for k in [3,2,1]:
    start=hold_start-pd.Timedelta(days=HORIZON*k)
    end=start+pd.Timedelta(days=HORIZON-1)
    if start>dev.date.min(): folds.append((start,end))
print("Rolling folds:",[(str(a.date()),str(b.date())) for a,b in folds])
print("Final holdout:",hold_dates.min(), "to",hold_dates.max())
assert len(folds)>=2

## 4. Candidate models and train-only preprocessing

The model comparison includes Ordinary Least Squares (OLS), Ridge, Lasso, Decision Tree, Random Forest, K-Nearest Neighbours (KNN), and LightGBM when the library is available. These methods provide comparisons across linear regression, single-tree, bagging, distance-based, and boosting approaches.

Numeric imputation and scaling, together with categorical imputation and one-hot encoding, are implemented inside scikit-learn pipelines. Consequently, preprocessing is fitted on each training fold rather than on validation data. Scaling is applied to linear models and KNN; tree-based models are used without scaling. LightGBM is included as the gradient-boosting candidate in the project methodology.

**Compute environment:** scikit-learn estimators run on the CPU. LightGBM uses the GPU only when a capability probe confirms that the installed build supports GPU execution; otherwise it runs on the CPU. Pandas feature engineering remains CPU-based.

**Why these candidate families?** OLS provides a simple linear reference; Ridge and Lasso test regularised linear relationships; a Decision Tree captures nonlinear splits; Random Forest reduces the variance of individual trees through bagging; KNN provides a distance-based comparison after scaling; and LightGBM uses gradient-boosted trees to capture nonlinear interactions in tabular calendar, store, promotion and lag features. LightGBM is treated as a candidate, not assumed to be best in advance.

**Configuration boundary.** Candidate configurations are fixed for this comparison rather than selected through a broad hyperparameter search. The experiment therefore compares the specified configurations; it does not establish that the selected model is globally optimal.


## GPU execution configuration

The notebook supports CPU and GPU environments. The installed LightGBM build is tested with a small fitting probe before GPU execution is selected. If the probe fails, LightGBM falls back to CPU execution. The presence of a GPU accelerator alone does not guarantee that the installed LightGBM package was compiled with GPU support.

In [ ]:
# Probe the installed LightGBM build instead of assuming GPU support.
LIGHTGBM_AVAILABLE = False
LIGHTGBM_DEVICE = "cpu"
try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
    try:
        _x = np.arange(200, dtype=float).reshape(-1, 1)
        _y = np.sin(_x[:, 0] / 10)
        _probe = LGBMRegressor(
            n_estimators=3, max_bin=63, device_type="gpu",
            verbosity=-1, n_jobs=-1, random_state=SEED
        )
        _probe.fit(_x, _y)
        LIGHTGBM_DEVICE = "gpu"
        print("LightGBM GPU probe succeeded. LightGBM will use GPU.")
    except Exception as gpu_error:
        LIGHTGBM_DEVICE = "cpu"
        print("LightGBM GPU unavailable in this runtime; using CPU.")
        print("Reason:", str(gpu_error)[:500])
except ImportError:
    print("LightGBM is not installed. Install it if you want the boosting comparison.")

print("LightGBM available:", LIGHTGBM_AVAILABLE, "| selected device:", LIGHTGBM_DEVICE)
try:
    import torch
    print("PyTorch CUDA visible:", torch.cuda.is_available())
except Exception:
    pass


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.base import clone

lags=[c for c in dev if c.startswith("lag_")]
rolls=[c for c in dev if c.startswith("roll_")]
num_cols=[c for c in ["onpromotion","promo_log1p","dow","day","month","week","quarter","year","weekend","dow_sin","dow_cos","month_sin","month_cos","national_holiday","regional_holiday","local_holiday"]+lags+rolls if c in dev]
cat_cols=[c for c in ["store_nbr","family","city","state","type","cluster"] if c in dev]
features=num_cols+cat_cols
def preprocessor(scale=True):
    ns=[("imputer",SimpleImputer(strategy="median"))]
    if scale: ns.append(("scale",StandardScaler()))
    return ColumnTransformer([
        ("num",Pipeline(ns),num_cols),
        ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),
                         ("ohe",OneHotEncoder(handle_unknown="ignore"))]),cat_cols)
    ])
def models():
    candidates = {
      "OLS":Pipeline([("prep",preprocessor()),("model",LinearRegression())]),
      "Ridge":Pipeline([("prep",preprocessor()),("model",Ridge(alpha=10))]),
      "Lasso":Pipeline([("prep",preprocessor()),("model",Lasso(alpha=.0005,max_iter=5000,random_state=SEED))]),
      "Decision Tree":Pipeline([("prep",preprocessor(False)),("model",DecisionTreeRegressor(max_depth=12,min_samples_leaf=50,random_state=SEED))]),
      "Random Forest":Pipeline([("prep",preprocessor(False)),("model",RandomForestRegressor(n_estimators=100,max_depth=18,min_samples_leaf=20,max_features=.7,n_jobs=-1,random_state=SEED))]),
      "KNN":Pipeline([("prep",preprocessor()),("model",KNeighborsRegressor(n_neighbors=30,weights="distance",n_jobs=-1))])
    }
    try:
        from lightgbm import LGBMRegressor
        candidates["LightGBM"] = Pipeline([
            ("prep", preprocessor(False)),
            ("model", LGBMRegressor(objective="regression", n_estimators=600,
                learning_rate=0.04, num_leaves=31, min_child_samples=30,
                subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                device_type=LIGHTGBM_DEVICE,
                max_bin=63 if LIGHTGBM_DEVICE == "gpu" else 255,
                random_state=SEED, n_jobs=-1, verbosity=-1))
        ])
    except ImportError:
        print("LightGBM not installed; install it to include the proposal's boosting candidate.")
    return candidates
def sample_fit(df):
    if MAX_TRAIN_ROWS is None or len(df)<=MAX_TRAIN_ROWS: return df
    return df.sample(MAX_TRAIN_ROWS,random_state=SEED).sort_index()
print("Numeric",len(num_cols),"categorical",cat_cols,"features",len(features))

## 5. Rolling-origin model comparison

The candidate models are compared using fixed initial configurations across the three development folds. A deterministic random sample of at most 80,000 training rows is used when a fold contains more rows; validation windows remain complete. This cap keeps the model comparison computationally manageable and is applied consistently to the candidate estimators, including LightGBM.

The reported results include fold-level RMSLE and MAE, along with the mean and standard deviation across folds. RMSLE is the primary comparison metric, while MAE provides an error measure in the original sales units.

The base model and all candidates are scored on identical validation windows. Mean rolling RMSLE is the primary model-selection metric; MAE is a complementary sales-scale measure. With only three folds, the fold standard deviation is descriptive evidence of temporal variation rather than a formal uncertainty interval or significance test.


In [ ]:
rows=[]; fold_preds={}
for fi,(start,end) in enumerate(folds,1):
    fit=dev[dev.date<start].dropna(subset=features+["log_sales"])
    val=dev[dev.date.between(start,end)]
    origin=start-pd.Timedelta(days=1)
    bp=naive_forecast(dev[dev.date<=origin],val,origin)
    rows.append({"fold":fi,"model":"Seasonal naive (Base model)","RMSLE":rmsle(val.sales,bp),"MAE":mae(val.sales,bp)})
    fold_preds[(fi,"Seasonal naive (Base model)")]=(val,bp)
    for name,est in models().items():
        fs=sample_fit(fit)
        est.fit(fs[features],fs.log_sales)
        pred=np.expm1(est.predict(val[features])).clip(0)
        rows.append({"fold":fi,"model":name,"RMSLE":rmsle(val.sales,pred),"MAE":mae(val.sales,pred)})
        fold_preds[(fi,name)]=(val,pred)
    print("Completed fold",fi,start.date(),end.date())
cv=pd.DataFrame(rows)
print("Fold-by-fold RMSLE (lower is better):")
display(cv.pivot(index="model",columns="fold",values="RMSLE").round(5))
print("Fold-by-fold MAE:")
display(cv.pivot(index="model",columns="fold",values="MAE").round(2))
summary=cv.groupby("model").agg(
    RMSLE_mean=("RMSLE","mean"), RMSLE_sd=("RMSLE","std"),
    MAE_mean=("MAE","mean"), MAE_sd=("MAE","std")
).sort_values("RMSLE_mean")
display(summary.round(5))
summary.RMSLE_mean.sort_values().plot(
    kind="bar", yerr=summary.loc[summary.RMSLE_mean.sort_values().index,"RMSLE_sd"].fillna(0),
    figsize=(10,4), title="Rolling-origin mean RMSLE (error bars: fold standard deviation)"
)
plt.ylabel("RMSLE (lower is better)")
plt.xticks(rotation=30,ha="right")
plt.tight_layout()
plt.show()


## 6. Final holdout evaluation

The model family with the lowest mean rolling-validation RMSLE is selected for the final holdout comparison. The selected model and Seasonal naive (Base model) are evaluated on the last 16 labelled days, 31 July–15 August 2017. These observations are kept separate from the rolling development folds.

The selected LightGBM model obtains a holdout RMSLE of 0.45270 and MAE of 85.77. The Seasonal naive (Base model) obtains a holdout RMSLE of 0.61704 and MAE of 96.54. The holdout RMSLE is higher than LightGBM's rolling-validation mean of 0.41428, indicating that this final period was more difficult for the fitted model under the selected configuration. The holdout is used for reporting rather than further model tuning.

In [ ]:
final_train=dev[dev.date<hold_start].dropna(subset=features+["log_sales"])
final_val=dev[dev.date.isin(hold_dates)]
origin=hold_start-pd.Timedelta(days=1)
final_preds={}
bp=naive_forecast(dev[dev.date<=origin],final_val,origin)
final_preds["Seasonal naive (Base model)"]=bp
final_rows=[{"model":"Seasonal naive (Base model)","RMSLE":rmsle(final_val.sales,bp),"MAE":mae(final_val.sales,bp)}]
for name,est in models().items():
    fs=sample_fit(final_train)
    est.fit(fs[features],fs.log_sales)
    p=np.expm1(est.predict(final_val[features])).clip(0)
    final_preds[name]=p
    final_rows.append({"model":name,"RMSLE":rmsle(final_val.sales,p),"MAE":mae(final_val.sales,p)})
final_metrics=pd.DataFrame(final_rows).sort_values("RMSLE")
display(final_metrics)
selected=summary.index[0]
print("CV-selected model:",selected)
display(final_metrics[final_metrics.model==selected])

## 6A. Forecast-horizon error analysis

A 16-day forecast can have different error characteristics at different lead times. This analysis reuses stored predictions from the three rolling-origin validation folds and compares the CV-selected model with the Seasonal naive (Base model) at each horizon step. It reports fold-level RMSLE, MAE and mean signed error, then summarises the three fold values. The final holdout is not used to select or tune a model.

In [ ]:

# Reuse predictions already computed in the three rolling-origin folds.
horizon_rows = []
horizon_models = list(dict.fromkeys([selected, "Seasonal naive (Base model)"]))
for fi, (start, end) in enumerate(folds, 1):
    for model_name in horizon_models:
        if (fi, model_name) not in fold_preds:
            continue
        validation, prediction = fold_preds[(fi, model_name)]
        prediction = np.asarray(prediction, dtype=float)
        horizon_step = (pd.to_datetime(validation["date"]) - pd.Timestamp(start)).dt.days + 1
        for step in range(1, HORIZON + 1):
            mask = horizon_step.to_numpy() == step
            if not mask.any():
                continue
            actual_step = validation.loc[mask, "sales"].to_numpy(dtype=float)
            predicted_step = prediction[mask]
            horizon_rows.append({
                "fold": fi, "forecast_horizon_day": step, "model": model_name,
                "n": int(mask.sum()),
                "RMSLE": rmsle(actual_step, predicted_step),
                "MAE": mae(actual_step, predicted_step),
                "mean_signed_error": float(np.mean(actual_step - predicted_step))
            })
horizon_error_by_fold = pd.DataFrame(horizon_rows)
if horizon_error_by_fold.empty:
    raise RuntimeError("No horizon-level predictions were available. Check fold_preds and model keys.")
horizon_error_summary = (
    horizon_error_by_fold.groupby(["forecast_horizon_day", "model"])
    .agg(
        n_total=("n", "sum"), RMSLE_mean=("RMSLE", "mean"), RMSLE_sd=("RMSLE", "std"),
        MAE_mean=("MAE", "mean"), MAE_sd=("MAE", "std"),
        mean_signed_error=("mean_signed_error", "mean")
    ).reset_index().sort_values(["model", "forecast_horizon_day"])
)
display(horizon_error_summary.round(4))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for model_name in horizon_models:
    part = horizon_error_summary[horizon_error_summary.model == model_name]
    if part.empty:
        continue
    ax[0].plot(part.forecast_horizon_day, part.RMSLE_mean, marker="o", label=model_name)
    ax[1].plot(part.forecast_horizon_day, part.MAE_mean, marker="o", label=model_name)
ax[0].set(title="Rolling RMSLE by forecast horizon", xlabel="Days ahead", ylabel="Mean fold RMSLE")
ax[1].set(title="Rolling MAE by forecast horizon", xlabel="Days ahead", ylabel="Mean fold MAE (sales units)")
for axis in ax:
    axis.set_xticks(range(1, HORIZON + 1)); axis.grid(alpha=.2); axis.legend()
plt.tight_layout(); plt.show()
for model_name in horizon_models:
    part = horizon_error_summary[horizon_error_summary.model == model_name]
    if not part.empty:
        worst_rmsle_day = int(part.loc[part.RMSLE_mean.idxmax(), "forecast_horizon_day"])
        worst_mae_day = int(part.loc[part.MAE_mean.idxmax(), "forecast_horizon_day"])
        print(f"{model_name}: highest mean RMSLE at day {worst_rmsle_day}; highest mean MAE at day {worst_mae_day}.")


## 7. Error analysis and interpretation

The holdout analysis compares predicted and observed sales, examines the residual distribution, and reports RMSLE and MAE by product family. These summaries show how aggregate performance varies across product categories and help identify series for which the selected model has larger errors.

In [ ]:
chosen=final_preds[selected]
err=final_val[["date","store_nbr","family","sales"]].copy()
err["prediction"]=chosen
err["residual"]=err.sales-err.prediction
err["abs_error"]=err.residual.abs()
err["sq_log_error"]=(np.log1p(err.sales)-np.log1p(err.prediction.clip(0)))**2
fig,ax=plt.subplots(1,2,figsize=(12,4))
lim=max(err.sales.quantile(.99),err.prediction.quantile(.99))
ax[0].scatter(err.sales,err.prediction,s=3,alpha=.15); ax[0].plot([0,lim],[0,lim],"--")
ax[0].set(xlim=(0,lim),ylim=(0,lim),title="Predicted vs actual (display capped at p99)",xlabel="Actual",ylabel="Predicted")
ax[1].hist(err.residual.clip(err.residual.quantile(.01),err.residual.quantile(.99)),bins=60)
ax[1].set(title="Residual distribution (1st–99th percentile)",xlabel="Actual - predicted")
plt.tight_layout(); plt.show()
family_error=err.groupby("family").apply(lambda g: pd.Series({"n":len(g),"RMSLE":np.sqrt(g.sq_log_error.mean()),"MAE":g.abs_error.mean()}),include_groups=False).sort_values("RMSLE",ascending=False)
display(family_error)
family_error.RMSLE.sort_values().plot(kind="barh",figsize=(9,9),title="Holdout RMSLE by product family"); plt.show()

### Store–family error analysis

The store–family audit ranks individual series by mean absolute error over the final holdout. Mean signed error indicates the direction of prediction error: positive values correspond to underprediction, while negative values correspond to overprediction. These results identify examples of larger errors but do not, by themselves, explain their causes.

In [ ]:
# Aggregate every store–family series in the final holdout.
# There are 54 stores × 33 families = 1,782 expected series in this panel.
store_family_error = (
    err.groupby(["store_nbr", "family"])
       .agg(
           n=("sales", "size"),
           actual_mean=("sales", "mean"),
           prediction_mean=("prediction", "mean"),
           signed_error_mean=("residual", "mean"),
           mean_absolute_error=("abs_error", "mean"),
           rmsle=("sq_log_error", lambda s: float(np.sqrt(s.mean())))
       )
       .reset_index()
       .sort_values("mean_absolute_error", ascending=False)
       .reset_index(drop=True)
)

# Keep a compact top-five table for display, but do not discard the full ranking.
worst_store_family = store_family_error.head(5).copy()

print("Store–family series in full holdout error table:", len(store_family_error))
print("Expected series from panel dimensions:", 54 * 33)
display(store_family_error.head(10).round(3))
display(worst_store_family.round(3))

# Change INSPECT_FAMILY to any family in final_val["family"] to inspect another series.
INSPECT_FAMILY = str(final_val["family"].mode().iloc[0])
family_slice = err[err.family == INSPECT_FAMILY].groupby("date").agg(
    actual=("sales", "sum"), prediction=("prediction", "sum")
).sort_index()
ax = family_slice.plot(figsize=(12, 4), marker="o",
                       title=f"Actual vs predicted daily sales — {INSPECT_FAMILY}")
ax.set_ylabel("Total sales across stores")
ax.set_xlabel("Holdout date")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
print("Inspected family:", INSPECT_FAMILY)


## 8. Feature-group ablation

Feature-group ablation compares the full feature set with variants that omit sales-history, calendar, promotion, holiday, or store-context features. Each variant uses the same rolling folds, model family, and fitting configuration. An increase in RMSLE after removing a feature group indicates that the group contributed predictive information under this evaluation design. The comparison is associational and is not interpreted as evidence of a causal effect.

In [ ]:
groups={
 "sales_history":[c for c in features if c.startswith("lag_") or c.startswith("roll_")],
 "calendar":[c for c in features if c in ["dow","day","month","week","quarter","year","weekend","dow_sin","dow_cos","month_sin","month_cos"]],
 "promotion":[c for c in features if c in ["onpromotion","promo_log1p"]],
 "holidays":[c for c in features if c in ["national_holiday","regional_holiday","local_holiday"]],
 "store_context":cat_cols
}
ablation=[]
ab_model=selected if selected!="Seasonal naive (Base model)" else summary.index[1]
for fi,(start,end) in enumerate(folds,1):
    fit=dev[dev.date<start].dropna(subset=features+["log_sales"]); val=dev[dev.date.between(start,end)]
    base=clone(models()[ab_model]); fs=sample_fit(fit); base.fit(fs[features],fs.log_sales)
    p=np.expm1(base.predict(val[features])).clip(0)
    ablation.append({"fold":fi,"feature_set":"Full","RMSLE":rmsle(val.sales,p)})
    for g,removed in groups.items():
        keep=[c for c in features if c not in removed]
        nkeep=[c for c in num_cols if c in keep]; ckeep=[c for c in cat_cols if c in keep]
        prep=ColumnTransformer([
          ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("scale",StandardScaler())]),nkeep),
          ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore"))]),ckeep)
        ])
        pipe=Pipeline([("prep",prep),("model",clone(models()[ab_model].named_steps["model"]))])
        pipe.fit(fs[keep],fs.log_sales)
        q=np.expm1(pipe.predict(val[keep])).clip(0)
        ablation.append({"fold":fi,"feature_set":"Without "+g,"RMSLE":rmsle(val.sales,q)})
ablation_df=pd.DataFrame(ablation)
display(ablation_df.groupby("feature_set").RMSLE.agg(["mean","std"]).sort_values("mean"))

## 9. Limitations, reproducibility, and conclusion

The experiments are based on one retailer and one historical period, so the results may not generalise to other retailers or market conditions. The 16-day horizon limits the use of recent target observations. In addition, the 80,000-row training cap may affect model rankings relative to fitting on all eligible rows. Aggregate metrics can conceal weak performance on particular store–family series. Feature ablation measures predictive contribution under the tested design, not causality. Finally, sales forecasts alone do not determine an inventory policy because stockout, spoilage, and holding-cost information is not supplied.

**Training-row cap.** When a training fold contains more than 80,000 eligible rows, a deterministic random sample is used for fitting. Validation observations are not sampled. The reported model rankings therefore describe the capped training setup and may differ from results obtained using the full training data.

**Evaluation interpretation.** LightGBM has the lowest mean rolling-validation RMSLE (0.41407 ± 0.00659) among the evaluated models. Its final holdout RMSLE is 0.45270, compared with 0.61704 for the Seasonal naive (Base model) baseline. This supports the use of LightGBM within the evaluated setup, while the higher holdout error indicates variation in performance across time periods.

**Feature contribution.** Removing sales-history features increases mean rolling RMSLE from 0.41428 to 0.58348, the largest deterioration among the tested ablations. Calendar, promotion, and store-context features also improve performance relative to their corresponding reduced feature sets. These findings describe predictive value in the evaluated folds and do not establish causal relationships.

**Reproducibility.** The notebook records the detected Python and package versions (NumPy, pandas, Matplotlib, scikit-learn, LightGBM and Jupyter) in the generated environment.json file. The competition CSV files are required in the detected data directory or in the directory specified by `DATA_DIR`. Running the notebook from top to bottom regenerates the analysis tables, figures, and files in `project_outputs/`.

**Conclusion.** Among the evaluated candidates, LightGBM provides the strongest rolling-origin RMSLE and improves on the Seasonal naive (Base model) baseline on the final holdout. Historical sales features provide the largest measured improvement in the ablation analysis. Performance nevertheless varies across product families and store–family series, and the training-row cap and limited historical scope should be considered when interpreting the results.

**Forecast-horizon interpretation.** The horizon-step analysis reuses rolling-fold predictions and reports errors separately for days 1–16. It is descriptive across the three folds; it does not provide calibrated prediction intervals and should not be interpreted as a guarantee for a future date.


### Exporting the full experiment outputs

The store–family error analysis exports **all evaluated store × family series** to `worst_store_family_errors.csv`, sorted by mean absolute error. The separate `top5_store_family_errors.csv` contains only the five largest-error examples. The export cell verifies that all expected files exist after writing them. In Kaggle, the files are saved under `/kaggle/working/project_outputs/` and can be downloaded from the notebook's output/files panel.


The updated notebook also exports forecast_horizon_error_by_fold.csv and forecast_horizon_error_analysis.csv. The latter is the 16-day summary for the CV-selected model and the Seasonal naive (Base model). Re-run the complete notebook to generate these files.


In [ ]:
# Save all experiment outputs to a predictable location in Kaggle.
# Kaggle notebooks normally expose /kaggle/working; local Jupyter falls back
# to a project_outputs folder under the current working directory.
KAGGLE_WORKING = Path("/kaggle/working")
OUT = (KAGGLE_WORKING / "project_outputs") if KAGGLE_WORKING.exists() else Path("project_outputs")
OUT.mkdir(parents=True, exist_ok=True)

cv.to_csv(OUT / "rolling_validation_results.csv", index=False)
summary.to_csv(OUT / "rolling_validation_summary.csv")
final_metrics.to_csv(OUT / "final_holdout_metrics.csv", index=False)
ablation_df.to_csv(OUT / "feature_ablation_results.csv", index=False)
horizon_error_by_fold.to_csv(OUT / "forecast_horizon_error_by_fold.csv", index=False)
horizon_error_summary.to_csv(OUT / "forecast_horizon_error_analysis.csv", index=False)
environment = {"python": platform.python_version(), "packages": PACKAGE_VERSIONS}
(OUT / "environment.json").write_text(json.dumps(environment, indent=2))
family_error.to_csv(OUT / "family_error_analysis.csv")

# This is the full store × family error ranking (up to 54 × 33 = 1,782 rows).
store_family_error.to_csv(OUT / "worst_store_family_errors.csv", index=False)
# Separate compact convenience file for the five largest MAE examples.
worst_store_family.to_csv(OUT / "top5_store_family_errors.csv", index=False)

err.sample(min(5000, len(err)), random_state=SEED).to_csv(
    OUT / "holdout_prediction_sample.csv", index=False
)
meta = {
    "project": "Retail Sales Forecasting and Feature Analysis",
    "course": "ES 335",
    "horizon_days": HORIZON,
    "folds": [
        {"start": str(a.date()), "end": str(b.date())}
        for a, b in folds
    ],
    "final_holdout": {
        "start": str(pd.Timestamp(hold_dates.min()).date()),
        "end": str(pd.Timestamp(hold_dates.max()).date())
    },
    "features": features,
    "max_train_rows": MAX_TRAIN_ROWS,
    "seed": SEED,
    "forecast_information_assumptions": {
        "historical_sales_lags_shifted_days": HORIZON,
        "calendar_features_known_in_advance": True,
        "promotion_values_assumed_known_in_advance": True,
        "future_actual_sales_used": False,
        "future_oil_and_transactions_used": False
    },
    "store_family_error_rows": int(len(store_family_error)),
    "store_family_error_sort": "mean_absolute_error descending",
    "environment": environment,
    "forecast_horizon_analysis": {"models": horizon_models, "validation_folds": len(folds), "horizon_days": HORIZON}
}
(OUT / "experiment_metadata.json").write_text(json.dumps(meta, indent=2))

# Verify that the intended files exist and are non-empty before finishing.
required_exports = [
    "rolling_validation_results.csv",
    "rolling_validation_summary.csv",
    "final_holdout_metrics.csv",
    "feature_ablation_results.csv",
    "forecast_horizon_error_by_fold.csv",
    "forecast_horizon_error_analysis.csv",
    "environment.json",
    "family_error_analysis.csv",
    "worst_store_family_errors.csv",
    "top5_store_family_errors.csv",
    "holdout_prediction_sample.csv",
    "experiment_metadata.json"
]
missing_exports = [name for name in required_exports if not (OUT / name).is_file()]
if missing_exports:
    raise FileNotFoundError(f"Output export(s) were not created: {missing_exports}")

print("Saved course-project outputs to:", OUT.resolve())
print("Full store-family error CSV:", (OUT / "worst_store_family_errors.csv").resolve())
print("Rows in full store-family error CSV:", len(store_family_error))
print("Verified exports:", ", ".join(required_exports))


## References and AI-use disclosure

- Kaggle, *Store Sales – Time Series Forecasting*: https://www.kaggle.com/competitions/store-sales-time-series-forecasting
- ES 335 course project guide and lecture plan.
- Python libraries: pandas, NumPy, Matplotlib, and scikit-learn; LightGBM for gradient boosting when available.

**AI-use disclosure.** AI assistance was used for project planning, code drafting, debugging, and editing. The project team is responsible for independently verifying the experiments, understanding the methods and results, and accurately describing the assistance used, in accordance with the course policy.

## Run-specific results summary

The following summary is generated from the executed experiment results. It reports the selected model, rolling-validation performance, final holdout comparison, training-row cap, and feature-ablation scores. These values are regenerated whenever the notebook is run.

In [ ]:
cv_best=summary.index[0]
cv_best_row=summary.loc[cv_best]
holdout_selected=final_metrics.loc[final_metrics.model==selected].iloc[0]
holdout_base=final_metrics.loc[final_metrics.model=="Seasonal naive (Base model)"].iloc[0]
ablation_summary=ablation_df.groupby("feature_set").RMSLE.mean().sort_values()
print(f"CV-selected model: {selected}")
print(f"Best mean rolling-validation RMSLE: {cv_best_row.RMSLE_mean:.5f} ± {cv_best_row.RMSLE_sd:.5f}")
print(f"Selected model final-holdout RMSLE: {holdout_selected.RMSLE:.5f}")
print(f"Selected model final-holdout MAE: {holdout_selected.MAE:.2f}")
print(f"Seasonal naive (Base model) final-holdout RMSLE: {holdout_base.RMSLE:.5f}")
print(f"Seasonal naive (Base model) final-holdout MAE: {holdout_base.MAE:.2f}")
print(f"Holdout RMSLE difference vs base model: {holdout_selected.RMSLE-holdout_base.RMSLE:+.5f}")
if selected in summary.index and "Seasonal naive (Base model)" in summary.index:
    print(f"Rolling MAE: selected model {summary.loc[selected, 'MAE_mean']:.2f} ± {summary.loc[selected, 'MAE_sd']:.2f}; base model {summary.loc['Seasonal naive (Base model)', 'MAE_mean']:.2f} ± {summary.loc['Seasonal naive (Base model)', 'MAE_sd']:.2f}")
print(f"Final holdout: {pd.Timestamp(hold_dates.min()).date()} to {pd.Timestamp(hold_dates.max()).date()}")
print(f"Training-row cap: {MAX_TRAIN_ROWS:,} rows per fitted model when needed")
print(f"Full-feature ablation mean rolling RMSLE: {ablation_summary['Full']:.5f}")
print(f"Without-sales-history ablation mean rolling RMSLE: {ablation_summary['Without sales_history']:.5f}")
display(ablation_summary.to_frame("mean_rolling_RMSLE").round(5))